# Question: 
## Implementation of K-Nearest Neighbors (K-NN) and Random Forest Classifiers

In [1]:
import numpy as np
import pandas as pd

In [2]:
data = pd.read_csv("heart_attack_dataset.csv")
data.head(2)

,Patient ID,Age,Sex,Cholesterol,Blood Pressure,Heart Rate,Diabetes,Family History,Smoking,Obesity,...,Sedentary Hours Per Day,Income,BMI,Triglycerides,Physical Activity Days Per Week,Sleep Hours Per Day,Country,Continent,Hemisphere,Heart Attack Risk
0,BMW7812,67,Male,208,158/88,72,0,0,1,0,...,6.615001,261404,31.251233,286,0,6,Argentina,South America,Southern Hemisphere,0
1,CZE1114,21,Male,389,165/93,98,1,1,1,1,...,4.963459,285768,27.194973,235,1,7,Canada,North America,Northern Hemisphere,0


In [3]:
temp = data["Blood Pressure"].apply(lambda x: np.int64(x.split("/")))
data["systolicBP"] = [x[0]for x in temp]
data["diastolicBP"] = [x[1]for x in temp]
data.drop("Blood Pressure",axis=1,inplace=True)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8763 entries, 0 to 8762
Data columns (total 27 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Patient ID                       8763 non-null   object 
 1   Age                              8763 non-null   int64  
 2   Sex                              8763 non-null   object 
 3   Cholesterol                      8763 non-null   int64  
 4   Heart Rate                       8763 non-null   int64  
 5   Diabetes                         8763 non-null   int64  
 6   Family History                   8763 non-null   int64  
 7   Smoking                          8763 non-null   int64  
 8   Obesity                          8763 non-null   int64  
 9   Alcohol Consumption              8763 non-null   int64  
 10  Exercise Hours Per Week          8763 non-null   float64
 11  Diet                             8763 non-null   object 
 12  Previous Heart Probl

In [4]:
data.drop("Patient ID",axis=1,inplace=True)
data.isnull().sum()

Age                                0
Sex                                0
Cholesterol                        0
Heart Rate                         0
Diabetes                           0
Family History                     0
Smoking                            0
Obesity                            0
Alcohol Consumption                0
Exercise Hours Per Week            0
Diet                               0
Previous Heart Problems            0
Medication Use                     0
Stress Level                       0
Sedentary Hours Per Day            0
Income                             0
BMI                                0
Triglycerides                      0
Physical Activity Days Per Week    0
Sleep Hours Per Day                0
Country                            0
Continent                          0
Hemisphere                         0
Heart Attack Risk                  0
systolicBP                         0
diastolicBP                        0
dtype: int64

In [5]:
numerical_columns = [x for x in data.columns if data[x].dtype=="int64" or data[x].dtype=="float64"]
categorical_columns = [x for x in data.columns if data[x].dtype=="object"]
categorical_columns

['Sex', 'Diet', 'Country', 'Continent', 'Hemisphere']

In [6]:
data[categorical_columns].nunique()

Sex            2
Diet           3
Country       20
Continent      6
Hemisphere     2
dtype: int64

In [7]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop="first",sparse_output=False)
x=ohe.fit_transform(data[categorical_columns])
x.shape

(8763, 28)

In [8]:
y=data[numerical_columns.pop(-3)]

In [9]:
xtrain = np.hstack((x,data[numerical_columns].to_numpy()))
xtrain.shape

(8763, 48)

# (KNN) k Nearest Neighbores

In [10]:
from sklearn.neighbors import KNeighborsClassifier
classifier = KNeighborsClassifier(n_neighbors = 3, metric = 'minkowski')
classifier.fit(xtrain,y)

KNeighborsClassifier(n_neighbors=3)

In [11]:
ypred=classifier.predict(xtrain)
ypred

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [12]:
from sklearn.metrics import classification_report
matrix = classification_report(y, ypred,labels=[1,0])

print('Classification report of non scaled model : \n',matrix)

Classification report of non scaled model : 
               precision    recall  f1-score   support

           1       0.71      0.59      0.64      3139
           0       0.79      0.87      0.83      5624

    accuracy                           0.77      8763
   macro avg       0.75      0.73      0.73      8763
weighted avg       0.76      0.77      0.76      8763



# Random Forest

In [13]:
from sklearn.ensemble import RandomForestClassifier
rfc=RandomForestClassifier(n_estimators=5)
rfc.fit(x,y)

RandomForestClassifier(n_estimators=5)

In [14]:
ypred = rfc.predict(x)
ypred

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [15]:
from sklearn.metrics import classification_report
matrix = classification_report(y, rfc.predict(x),labels=y.unique())

print('Classification report of non scaled model : \n',matrix)

Classification report of non scaled model : 
               precision    recall  f1-score   support

           0       0.64      1.00      0.78      5624
           1       0.45      0.01      0.01      3139

    accuracy                           0.64      8763
   macro avg       0.54      0.50      0.40      8763
weighted avg       0.57      0.64      0.51      8763



# Conclusion: Heart Attack Risk Classification

Based on the modeling performed in the notebook, here are the key findings and conclusions regarding the prediction of Heart Attack Risk using K-Nearest Neighbors (KNN) and Random Forest classifiers.

## 1. Data Preprocessing
* **Feature Engineering:** The raw `Blood Pressure` string column was successfully parsed into distinct numerical `systolicBP` and `diastolicBP` features, which is essential for mathematical modeling.
* **Categorical Encoding:** One-Hot Encoding was correctly applied to categorical features (`Sex`, `Diet`, `Country`, `Continent`, and `Hemisphere`) to convert them into a machine-readable format.

## 2. Model Performance Comparison
* **K-Nearest Neighbors (KNN):**
    * The KNN model (`k=3`) achieved an overall accuracy of **77%**.
    * It showed a reasonable balance in detecting both classes, capturing an F1-score of 0.83 for Class 0 (No Risk) and 0.64 for Class 1 (High Risk). 
* **Random Forest:**
    * The Random Forest model (`n_estimators=5`) achieved an overall accuracy of **64%**.
    * **Important Observation:** The Random Forest classifier struggled significantly to identify Class 1, yielding a recall of just 0.01. This severe underperformance is directly tied to an implementation anomaly in the notebook: the Random Forest model was fitted *only* on the one-hot encoded categorical features (`x`) rather than the complete feature array (`xtrain`) that was used for the KNN model.

## 3. Key Limitations & Future Improvements
To make these models reliable for real-world medical predictions, the following improvements are highly recommended:
* **Train/Test Split Implementation:** Currently, both models are trained and evaluated on the exact same dataset, which leads to artificial data leakage and overfitting. Implementing a standard `train_test_split` is critical to evaluate how well the models generalize to unseen patient data.
* **Feature Scaling:** KNN is a distance-based algorithm. The dataset contains features with vastly different scales (e.g., `Income` in the hundreds of thousands vs. `Sleep Hours`). Applying standard scaling (e.g., `StandardScaler`) prior to modeling is necessary to prevent features with larger magnitudes from dominating the distance calculations.
* **Correcting the Random Forest Input:** The Random Forest model must be retrained using the complete combined feature set (`xtrain`). Additionally, increasing the number of estimators (trees) from 5 to a standard baseline (e.g., 100) will provide a much more stable and accurate ensemble model.